# 01 · Silver + features (past only)
Attach **lh_fraud** as default lakehouse. Run with `SAMPLE = True` first (300 customers, cheap), then once with `SAMPLE = False`.

Output tables: `silver_transactions`, `gold_features`.
Rule: every feature uses only rows **before** the transaction (`rowsBetween(unboundedPreceding, -1)`, `rangeBetween(-secs, -1)`).

In [ ]:
from pyspark.sql import functions as F, Window as W

SAMPLE = True   # True = 300 customers to test the notebook cheaply. False = full run (do it once).
spark.conf.set("spark.sql.shuffle.partitions", "32" if SAMPLE else "64")

## Silver: one clean, joined transaction table (train + test)

In [ ]:
tx = spark.table("v_transactions_all").drop("split")
if SAMPLE:
    keep = spark.table("customers").select("customer_id").orderBy(F.rand(42)).limit(300)
    tx = tx.join(keep, "customer_id")

labels = spark.table("fraud_labels_train")
customers = spark.table("customers").drop("gender")   # gender deliberately not used (fairness)
cards = spark.table("cards").select("card_id", "card_type", "is_international", "daily_limit_tnd",
                                    "atm_daily_limit_tnd", "contactless_enabled")
merchants = spark.table("merchants").select("merchant_id", "merchant_type")

silver = (tx.dropDuplicates(["transaction_id"])
            .join(labels, "transaction_id", "left")
            .join(customers, "customer_id", "left")
            .join(cards, "card_id", "left")
            .join(merchants, "merchant_id", "left")
            .withColumn("amount_tnd", F.col("amount_tnd").cast("double"))
            .withColumn("latitude", F.col("latitude").cast("double"))
            .withColumn("longitude", F.col("longitude").cast("double"))
            .withColumn("ts", F.col("timestamp").cast("long"))
            .withColumn("period", F.when(F.month("timestamp") <= 5, "train")
                                   .when(F.month("timestamp") <= 7, "valid").otherwise("test")))

silver.write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable("silver_transactions")
s = spark.table("silver_transactions")
display(s.groupBy("period").agg(F.count("*").alias("rows"), F.sum("is_fraud").alias("frauds"),
                                F.round(F.avg("is_fraud") * 100, 3).alias("fraud_pct")).orderBy("period"))

## Features: amount vs habits, velocity, novelty, geography, channel, time, profile

In [ ]:
s = spark.table("silver_transactions")
HOME = s.groupBy("country").count().orderBy(F.desc("count")).first()["country"]
print("home country value in data:", HOME)

past = W.partitionBy("customer_id").orderBy("ts").rowsBetween(W.unboundedPreceding, -1)
by_cust = W.partitionBy("customer_id").orderBy("ts")

def rng(key, secs):
    return W.partitionBy(key).orderBy("ts").rangeBetween(-secs, -1)

def first_time(*cols):
    w = W.partitionBy("customer_id", *cols).orderBy("ts").rowsBetween(W.unboundedPreceding, -1)
    return (F.count(F.lit(1)).over(w) == 0).cast("int")

def haversine(lat1, lon1, lat2, lon2):
    dlat, dlon = F.radians(lat2 - lat1), F.radians(lon2 - lon1)
    a = F.pow(F.sin(dlat / 2), 2) + F.cos(F.radians(lat1)) * F.cos(F.radians(lat2)) * F.pow(F.sin(dlon / 2), 2)
    return 2 * 6371 * F.asin(F.sqrt(a))

phys = F.col("channel").isin("TPE", "DAB")
foreign = F.coalesce(F.col("country") != HOME, F.lit(False))

f = (s.withColumn("key", F.coalesce("card_id", "customer_id"))          # card if any, else customer (transfers)
      .withColumn("is_physical", phys.cast("int"))
      .withColumn("f_declined", (F.col("status") == "REFUSEE").cast("int"))
      .withColumn("f_hour", F.hour("timestamp"))
      .withColumn("f_is_night", (F.col("f_hour") < 6).cast("int"))
      .withColumn("f_dow", F.dayofweek("timestamp"))
      .withColumn("f_payday", ((F.dayofmonth("timestamp") >= 25) | (F.dayofmonth("timestamp") <= 3)).cast("int"))
      .withColumn("f_ramadan", F.to_date("timestamp").between("2026-02-18", "2026-03-19").cast("int"))
      # --- customer history BEFORE this transaction
      .withColumn("f_n_before", F.count(F.lit(1)).over(past))
      .withColumn("f_avg_amt_before", F.avg("amount_tnd").over(past))
      .withColumn("f_std_amt_before", F.stddev("amount_tnd").over(past))
      .withColumn("f_max_amt_before", F.max("amount_tnd").over(past))
      .withColumn("f_night_share_before", F.avg("f_is_night").over(past))
      .withColumn("f_secs_since_prev", F.col("ts") - F.lag("ts").over(by_cust))
      # --- amount
      .withColumn("f_amount", F.col("amount_tnd"))
      .withColumn("f_amount_ratio", F.col("amount_tnd") / F.col("f_avg_amt_before"))
      .withColumn("f_amount_z", (F.col("amount_tnd") - F.col("f_avg_amt_before")) / F.col("f_std_amt_before"))
      .withColumn("f_amount_over_max", F.col("amount_tnd") / F.col("f_max_amt_before"))
      .withColumn("f_amount_over_income", F.col("amount_tnd") / F.col("monthly_income_tnd").cast("double"))
      .withColumn("f_amount_over_limit", F.col("amount_tnd") / F.when(F.col("channel") == "DAB", F.col("atm_daily_limit_tnd"))
                                                              .otherwise(F.col("daily_limit_tnd")).cast("double"))
      # --- velocity (past only)
      .withColumn("f_key_tx_10m", F.count(F.lit(1)).over(rng("key", 600)))
      .withColumn("f_key_tx_1h", F.count(F.lit(1)).over(rng("key", 3600)))
      .withColumn("f_key_declines_1h", F.sum("f_declined").over(rng("key", 3600)))
      .withColumn("f_key_small_tx_1h", F.sum((F.col("amount_tnd") < 20).cast("int")).over(rng("key", 3600)))
      .withColumn("f_cust_tx_24h", F.count(F.lit(1)).over(rng("customer_id", 86400)))
      .withColumn("f_cust_amt_24h", F.sum("amount_tnd").over(rng("customer_id", 86400)))
      # --- novelty
      .withColumn("f_new_merchant", F.when(F.col("merchant_id").isNull(), 0).otherwise(first_time("merchant_id")))
      .withColumn("f_new_device", F.when(F.col("device_id").isNull(), 0).otherwise(first_time("device_id")))
      .withColumn("f_new_beneficiary", F.when(F.col("beneficiary_id").isNull(), 0).otherwise(first_time("beneficiary_id")))
      .withColumn("f_new_ip_country", F.when(F.col("ip_country").isNull(), 0).otherwise(first_time("ip_country")))
      .withColumn("f_new_city_phys", F.when(phys & F.col("city").isNotNull(), first_time("is_physical", "city")).otherwise(0))
      .withColumn("f_new_country_phys", F.when(phys, first_time("is_physical", "country")).otherwise(0))
      .withColumn("f_is_foreign", foreign.cast("int"))
      .withColumn("f_ip_foreign", F.coalesce(F.col("ip_country") != HOME, F.lit(False)).cast("int"))
      .withColumn("f_same_country_7d", F.count(F.lit(1)).over(
          W.partitionBy("customer_id", "country").orderBy("ts").rangeBetween(-7 * 86400, -1)))
      # --- geography: distance / speed since previous located transaction
      .withColumn("prev_lat", F.last("latitude", ignorenulls=True).over(past))
      .withColumn("prev_lon", F.last("longitude", ignorenulls=True).over(past))
      .withColumn("prev_geo_ts", F.last(F.when(F.col("latitude").isNotNull(), F.col("ts")), ignorenulls=True).over(past))
      .withColumn("f_dist_prev_km", haversine(F.col("prev_lat"), F.col("prev_lon"), F.col("latitude"), F.col("longitude")))
      .withColumn("f_speed_kmh", F.col("f_dist_prev_km") /
                  F.greatest((F.col("ts") - F.col("prev_geo_ts")) / 3600.0, F.lit(0.05)))
      # --- channel / security
      .withColumn("f_magstripe", (F.col("entry_mode") == "MAGSTRIPE").cast("int"))
      .withColumn("f_contactless", (F.col("entry_mode") == "SANS_CONTACT").cast("int"))
      .withColumn("f_ecom_no3ds", (F.col("entry_mode") == "ECOM_SANS_3DS").cast("int"))
      .withColumn("f_chip_abroad", ((F.col("entry_mode") == "CHIP") & foreign).cast("int"))
      .withColumn("f_contactless_near_limit", ((F.col("entry_mode") == "SANS_CONTACT") &
                                               F.col("amount_tnd").between(150, 200)).cast("int"))
      .withColumn("f_card_not_intl_abroad", ((F.col("is_international") == False) & foreign).cast("int"))
      .withColumn("f_unusual_night", ((F.col("f_is_night") == 1) & (F.col("f_night_share_before") < 0.05) &
                                      (F.col("f_n_before") >= 10)).cast("int"))
      # --- profile
      .withColumn("f_age", F.col("age").cast("double"))
      .withColumn("f_income", F.col("monthly_income_tnd").cast("double"))
      .withColumn("f_balance", F.col("avg_monthly_balance_tnd").cast("double"))
      .withColumn("f_tenure_days", F.datediff(F.to_date("timestamp"), F.col("customer_since")))
      .withColumn("f_intl_card", F.col("is_international").cast("int"))
      .withColumn("f_digital_user", F.col("digital_banking_user").cast("int"))
      .drop("prev_lat", "prev_lon", "prev_geo_ts"))
f = f.cache()
print("rows:", f.count())

## Digital-banking signals in the 24 h before MOBILE / WEB / transfer operations

In [ ]:
ev = (spark.table("digital_events")
        .select(F.col("customer_id").alias("ev_cust"), F.col("timestamp").cast("long").alias("ets"),
                "event_type", F.col("status").alias("ev_status"), F.col("channel").alias("ev_channel"),
                F.col("beneficiary_id").alias("ev_benef")))
t = (f.filter(F.col("channel").isin("MOBILE", "WEB") | (F.col("transaction_type") == "VIREMENT"))
      .select("transaction_id", "customer_id", "ts", "beneficiary_id"))

def n(cond):
    return F.sum(cond.cast("int"))

dig = (t.join(ev, (F.col("customer_id") == F.col("ev_cust")) & (F.col("ets") < F.col("ts")) &
                  (F.col("ets") >= F.col("ts") - 86400))
        .groupBy("transaction_id")
        .agg(n((F.col("event_type") == "CONNEXION") & (F.col("ev_status") == "ECHEC")).alias("f_login_fail_24h"),
             n(F.col("event_type") == "AJOUT_APPAREIL").alias("f_device_added_24h"),
             n(F.col("event_type") == "AJOUT_BENEFICIAIRE").alias("f_benef_added_24h"),
             n(F.col("event_type") == "REINITIALISATION_MOT_DE_PASSE").alias("f_pwd_reset_24h"),
             n(F.col("event_type") == "MODIFICATION_NUMERO_TELEPHONE").alias("f_phone_change_24h"),
             n(F.col("ev_channel").isin("AGENCE", "CENTRE_APPEL")).alias("f_branch_events_24h"),
             F.min(F.when((F.col("event_type") == "AJOUT_BENEFICIAIRE") & (F.col("ev_benef") == F.col("beneficiary_id")),
                          (F.col("ts") - F.col("ets")) / 60.0)).alias("f_mins_since_benef_added")))

## Graph signals: merchant / device shared with frauds **already reported** before the transaction
Only `fraud_reported_date < transaction date` is used (the one legitimate use of that column, brief § 7.1).

In [ ]:
reported = s.filter(F.col("is_fraud") == 1).select("merchant_id", "device_id", F.col("fraud_reported_date").alias("rdate"))

def shared(col, name):
    r = (reported.filter(F.col(col).isNotNull()).groupBy(F.col(col).alias("g_key"), "rdate")
                 .agg(F.count(F.lit(1)).alias("n")))
    base = f.filter(F.col(col).isNotNull()).select("transaction_id", F.col(col).alias("t_key"),
                                                   F.to_date("timestamp").alias("d"))
    return (base.join(r, (F.col("t_key") == F.col("g_key")) & (F.col("rdate") < F.col("d")) &
                         (F.col("rdate") >= F.date_sub(F.col("d"), 30)))
                .groupBy("transaction_id").agg(F.sum("n").alias(name)))

COUNTS = ["f_login_fail_24h", "f_device_added_24h", "f_benef_added_24h", "f_pwd_reset_24h", "f_phone_change_24h",
          "f_branch_events_24h", "f_merchant_frauds_30d", "f_device_frauds_30d"]
gold = (f.join(dig, "transaction_id", "left")
         .join(shared("merchant_id", "f_merchant_frauds_30d"), "transaction_id", "left")
         .join(shared("device_id", "f_device_frauds_30d"), "transaction_id", "left")
         .fillna(0, subset=COUNTS))

CONTEXT = ["transaction_id", "timestamp", "period", "customer_id", "card_id", "merchant_id", "beneficiary_id",
           "device_id", "transaction_type", "channel", "entry_mode", "merchant_category", "amount_tnd", "city",
           "country", "ip_country", "status", "decline_reason", "segment", "kyc_risk_level", "card_type",
           "merchant_type", "home_city", "is_fraud", "fraud_scenario"]
FEATS = [c for c in gold.columns if c.startswith("f_")]
gold = gold.select(*CONTEXT, *[F.col(c).cast("double").alias(c) for c in FEATS])
gold.write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable("gold_features")
print(len(FEATS), "features:", FEATS)
f.unpersist()

## Quick sanity check: fraud rate when each binary signal fires (train period)

In [ ]:
g = spark.table("gold_features").filter(F.col("period") == "train")
flags = ["f_new_country_phys", "f_magstripe", "f_unusual_night", "f_card_not_intl_abroad", "f_contactless_near_limit",
         "f_new_device", "f_new_beneficiary", "f_ip_foreign"]
display(spark.createDataFrame(
    [(c, *g.filter(F.col(c) == 1).agg(F.count(F.lit(1)), F.avg("is_fraud")).first()) for c in flags],
    "signal string, rows long, fraud_rate double").orderBy(F.desc("fraud_rate")))

In [ ]:
mssparkutils.session.stop()   # free the capacity